# DreamerV3 Smoke Test — HM3D Easy Episodes

**Goal:** Verify the DreamerV3 training pipeline works end-to-end on HM3D ObjectNav.
This is NOT a full training run — just a quick validation (~5-10 min on `dev_gpu_h100`).

**What we verify:**
1. Environment loads and filters easy episodes (geodesic < 5m)
2. Replay buffer fills correctly
3. Training loop runs, losses decrease
4. Save/load checkpoint works
5. Greedy eval produces actions

**Config:** 64×64 obs, 5k training steps, batch_size=4, seq_len=16

**Run on bwUniCluster:**
```bash
srun --partition=dev_gpu_h100 --gres=gpu:1 --time=00:30:00 \
  uv run jupyter nbconvert --to notebook --execute notebooks/dreamerv3_smoke_test.ipynb --inplace
```

In [ ]:
import os, sys
from pathlib import Path

# Ensure project root is on path
project_root = Path(os.path.abspath("")).parent if Path(os.path.abspath("")).name == "notebooks" else Path(os.path.abspath(""))
os.chdir(project_root)
print(f"Working directory: {Path.cwd()}")

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from src.dreamerv3.configs import DreamerConfig
from src.dreamerv3.agent import DreamerAgent
from src.dreamerv3.replay_buffer import ReplayBuffer

print(f"JAX devices: {jax.devices()}")
print(f"JAX backend: {jax.default_backend()}")

In [ ]:
# Tiny config for quick validation
config = DreamerConfig(
    obs_shape=(3, 64, 64),
    max_episode_steps=200,
    split="val_mini",
    total_steps=5000,
    prefill_steps=500,
    batch_size=4,
    seq_len=16,
    save_every=2500,
    log_every=100,
    buffer_capacity=10_000,
    logdir="output/dreamerv3-smoke",
    seed=42,
)

print("Config:")
for f in ['obs_shape', 'max_episode_steps', 'split', 'total_steps',
          'prefill_steps', 'batch_size', 'seq_len', 'hidden_size', 'stoch_size']:
    print(f"  {f}: {getattr(config, f)}")

In [ ]:
# Create environment with easy episode filter
from src.dreamerv3.env_habitat import HabitatObjectNavEnv

env = HabitatObjectNavEnv(config, max_geodesic=5.0)

# Verify
obs = env.reset()
print(f"\nFirst observation:")
print(f"  image shape: {obs['image'].shape} dtype: {obs['image'].dtype}")
print(f"  is_first: {obs['is_first']}")
print(f"  Episode: {env._env.current_episode.episode_id}")
print(f"  Target: {env._env.current_episode.object_category}")
print(f"  Geodesic: {env._env.current_episode.info.get('geodesic_distance', '?'):.2f}m")

In [ ]:
# Prefill buffer with random actions
buffer = ReplayBuffer(config)

obs = env.reset()
for i in range(config.prefill_steps):
    action = np.random.randint(0, config.num_actions)
    next_obs = env.step(action)
    buffer.add(obs["image"], action, next_obs["reward"], next_obs["done"])
    obs = next_obs if not next_obs["done"] else env.reset()

print(f"Buffer size after prefill: {buffer.size}")
print(f"Buffer capacity: {buffer.capacity}")

# Verify sampling works
batch = buffer.sample(config.batch_size, config.seq_len)
print(f"\nSample batch shapes:")
for k, v in batch.items():
    print(f"  {k}: {v.shape} {v.dtype}")

In [ ]:
# Train for 5k steps — track metrics
rng_key = jax.random.PRNGKey(config.seed)
rng_key, init_key = jax.random.split(rng_key)
agent = DreamerAgent(config, init_key)

obs = env.reset()
episode_reward = 0.0
episode_count = 0
metrics_log = []  # (step, metrics_dict)
episode_log = []  # (step, reward, success)

print(f"Training for {config.total_steps} steps...")
for step in range(config.total_steps):
    rng_key, act_key, train_key = jax.random.split(rng_key, 3)

    action = agent.act(obs, act_key)
    next_obs = env.step(action)
    buffer.add(obs["image"], action, next_obs["reward"], next_obs["done"])
    episode_reward += next_obs["reward"]

    if next_obs["done"]:
        episode_count += 1
        episode_log.append((step, episode_reward, next_obs.get("success", 0.0)))
        episode_reward = 0.0
        obs = env.reset()
    else:
        obs = next_obs

    if buffer.size >= config.batch_size * config.seq_len:
        batch = buffer.sample(config.batch_size, config.seq_len)
        metrics = agent.train_step(batch, train_key)

        if step % config.log_every == 0:
            metrics_log.append((step, {k: float(v) for k, v in metrics.items()}))
            print(f"  [step {step:>5}] wm={metrics['wm_loss']:.3f} "
                  f"actor={metrics['actor_loss']:.3f} critic={metrics['critic_loss']:.3f} "
                  f"ep={episode_count}")

    if step > 0 and step % config.save_every == 0:
        agent.save(config.logdir)
        print(f"  [step {step}] Checkpoint saved")

agent.save(config.logdir)
print(f"\nDone. {episode_count} episodes, {len(metrics_log)} metric snapshots.")

In [ ]:
# Verify save/load
rng_key, reload_key = jax.random.split(rng_key)
agent2 = DreamerAgent(config, reload_key)
agent2.load(config.logdir)

# Compare params
orig_flat = jax.tree.leaves(agent.wm_state.params)
loaded_flat = jax.tree.leaves(agent2.wm_state.params)
all_match = all(jnp.allclose(a, b) for a, b in zip(orig_flat, loaded_flat))
print(f"Params match after save/load: {all_match}")
assert all_match, "Checkpoint save/load mismatch!"

In [ ]:
# Quick greedy eval (5 episodes)
print("Greedy eval (5 episodes):")
eval_results = []
for ep in range(5):
    obs = env.reset()
    ep_reward, steps = 0.0, 0
    rng_key, act_key = jax.random.split(rng_key)

    while not obs.get("done", False) and steps < config.max_episode_steps:
        action = agent2.act(obs, act_key, training=False)
        obs = env.step(action)
        ep_reward += obs["reward"]
        steps += 1

    cat = getattr(env._env.current_episode, "object_category", "?")
    print(f"  ep {ep+1}: reward={ep_reward:+.2f} success={obs.get('success',0):.0f} "
          f"steps={steps} target={cat}")
    eval_results.append({"reward": ep_reward, "success": obs.get("success", 0), "steps": steps})

env.close()
print(f"\nMean reward: {np.mean([r['reward'] for r in eval_results]):.2f}")
print(f"SR: {np.mean([r['success'] for r in eval_results]):.2f}")
print("(SR~0 expected after only 5k steps — this just verifies the pipeline works)")

In [ ]:
# Plot training curves
if metrics_log:
    steps_arr = [m[0] for m in metrics_log]
    wm = [m[1]["wm_loss"] for m in metrics_log]
    actor = [m[1]["actor_loss"] for m in metrics_log]
    critic = [m[1]["critic_loss"] for m in metrics_log]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(steps_arr, wm, color="#4C72B0")
    axes[0].set_title("World Model Loss")
    axes[0].set_xlabel("Step")

    axes[1].plot(steps_arr, actor, color="#DD8452")
    axes[1].set_title("Actor Loss")
    axes[1].set_xlabel("Step")

    axes[2].plot(steps_arr, critic, color="#55A868")
    axes[2].set_title("Critic Loss")
    axes[2].set_xlabel("Step")

    plt.suptitle("DreamerV3 Smoke Test — 5k Steps", y=1.02)
    plt.tight_layout()
    plt.show()

    # Episode rewards
    if episode_log:
        ep_steps = [e[0] for e in episode_log]
        ep_rewards = [e[1] for e in episode_log]
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.plot(ep_steps, ep_rewards, 'o-', markersize=3, alpha=0.7)
        ax.set_xlabel("Step")
        ax.set_ylabel("Episode Reward")
        ax.set_title(f"Episode Rewards ({len(episode_log)} episodes)")
        plt.tight_layout()
        plt.show()

print("Smoke test complete. All systems operational.")